In [1]:
import torch
import torch.nn as nn
from datasets import load_dataset
from torch.utils.data import DataLoader, random_split
from torchtext.data.utils import get_tokenizer
from collections import defaultdict
from torch.nn.utils.rnn import pad_sequence
from collections import Counter
from torchtext.vocab import GloVe
from torchtext.vocab import vocab as Vocab
from torch.cuda.amp import GradScaler,autocast

dataset = load_dataset("imdb")

train_data = dataset["train"]
test_data  = dataset["test"]

print(Counter(train_data["label"]))  
print(Counter(test_data["label"])) 


/home/aayam/micromamba/envs/RNN_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/aayam/micromamba/envs/RNN_env/lib/python3.11/site-packages/torchtext/data/__init__.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last released version will be 0.18 (this one). You can silence this warning by calling the following at the beginnign of your scripts: `import torchtext; torchtext.disable_torchtext_deprecation_warning()`
  warnings.warn(torchtext._TORCHTEXT_DEPRECATION_MSG)
/home/aayam/micromamba/envs/RNN_env/lib/python3.11/site-packages/torchtext/vocab/__init__.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last released version will be 0.18 (this one). You can silence this warning by calling t

Counter({0: 12500, 1: 12500})
Counter({0: 12500, 1: 12500})


In [2]:
tokenizer= get_tokenizer("basic_english")

all_tokens = []
for text in dataset["train"]["text"]:
    all_tokens.extend(tokenizer(text))

counter = Counter(all_tokens)
specials= ["<unk>"]
vocab = Vocab(counter, specials=specials)
vocab.set_default_index(vocab["<unk>"])

vocab_size= len(vocab) 

def text_pipeline(text, vocab=vocab):

   return [vocab[token] for token in tokenizer(text)]
    

In [15]:
glove_vectors= GloVe(name='6B', dim=200)

embedding_dim=200


embedding_layer= nn.Embedding(vocab_size, embedding_dim)

weights_matrix = torch.zeros((vocab_size, embedding_dim))


itos = vocab.get_itos()

for i, word in enumerate(itos):
    try:
        weights_matrix[i] = glove_vectors[word]
    except KeyError:
        weights_matrix[i] = torch.randn(embedding_dim)

embedding_layer.weight.data.copy_(weights_matrix)
embedding_layer.weight.requires_grad = False


In [16]:
class SentimentClassifier(nn.Module):
    def __init__(self,vocab_size,embedding_dim,hidden_dim,output_dim):
        super().__init__()

        self.embedding= embedding_layer
        self.lstm= nn.LSTM(embedding_dim, hidden_dim, batch_first=True, dropout=0.2, bidirectional=True, num_layers=2)
        self.dropout= nn.Dropout(0.3)
        self.fc= nn.Linear(hidden_dim*2, output_dim)
    
    def forward(self, text):

        embedding= self.embedding(text)
        output, (h_n,c_n)= self.lstm(embedding)
        forward_hidden_state= h_n[-2]
        backward_hidden_state= h_n[-1]
        last_hidden_state = torch.cat([forward_hidden_state, backward_hidden_state], dim=1)
        last_hidden_state= self.dropout(last_hidden_state)
        out= self.fc(last_hidden_state)

        return out 

device= torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model= SentimentClassifier(vocab_size=vocab_size,embedding_dim=200, hidden_dim=128, output_dim=1)
model= model.to(device)


In [10]:
def collate(batch):

    text_list, label_list= [],[]
    max_len= 256

    for item in batch:

        label_list.append(item['label'])
        tokens = text_pipeline(item["text"], vocab)
        if not tokens:
            tokens=[0]
        processed_Text= torch.tensor(tokens[:max_len], dtype= torch.int64)
        text_list.append(processed_Text)

    text_list= pad_sequence(text_list, batch_first=True, padding_value=0)
    label_list= torch.tensor(label_list)

    return label_list, text_list

train_size= int(0.9 * len(train_data))
val_size= len(train_data)- train_size

train_set, val_set= random_split(train_data,[train_size,val_size])

data_loader= DataLoader(train_set, batch_size=64,shuffle=True, collate_fn=collate)
test_loader= DataLoader(test_data,batch_size=64,shuffle=False, collate_fn=collate)
val_loader= DataLoader(val_set,batch_size=64,shuffle=False, collate_fn=collate)

In [13]:
def train_step(model,batch,optimizer,scaler,criterion):
    label,text= batch
    label= label.float().unsqueeze(1).to(device)
    text=  text.to(device)

    with autocast():
        output= model(text)
        loss= criterion(output, label)

    optimizer.zero_grad()
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

    return loss.item()


def val_step(model,batch,criterion):
    label,text= batch
    label= label.float().unsqueeze(1).to(device)
    text= text.to(device)

    with autocast():
        output= model(text)
        loss= criterion(output,label)

    return loss.item()

    
    

In [17]:
optimizer= torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
criterion= nn.BCEWithLogitsLoss()
scheduler= torch.optim.lr_scheduler.CosineAnnealingLR(optimizer=optimizer, T_max=20, eta_min=1e-6)

num_epochs=20
counter=0
patience=3

best_loss= float("inf")

scaler= GradScaler()

for epoch in range(num_epochs):

    model.train()
    running_loss=0

    for batch_idx, batch in enumerate(data_loader):

        loss= train_step(model,batch,optimizer,scaler,criterion)
        running_loss+=loss

        if (batch_idx+1) % 100 ==0:
            print(f"epoch {epoch+1}/{num_epochs} | batch {batch_idx+1}/{len(data_loader)} | loss {loss:.4f}")
   
    average_loss= running_loss/len(data_loader)
    print(f"average loss of epoch {epoch+1}/{num_epochs} | : {average_loss}")
    
    with torch.no_grad():
        model.eval()
        running_val_loss=0

        for batch in val_loader:

            loss= val_step(model,batch,criterion)
            running_val_loss+= loss

    average_val_loss= running_val_loss/ len(val_loader)
    print(f"average validation loss of epoch {epoch+1}/{num_epochs} | : {average_val_loss}")

    scheduler.step()

    if average_val_loss < best_loss:
        best_loss= average_val_loss
        counter=0
        torch.save(model.state_dict(), "best_parameters.pth")

    else:
        counter+=1
        if counter>=patience:
              print(f"Early stopping at epoch {epoch+1}")
              break



epoch 1/20 | batch 100/352 | loss 0.7004
epoch 1/20 | batch 200/352 | loss 0.6389
epoch 1/20 | batch 300/352 | loss 0.6766
average loss of epoch 1/20 | : 0.6778810769319534
average validation loss of epoch 1/20 | : 0.6684983238577843
epoch 2/20 | batch 100/352 | loss 0.7209
epoch 2/20 | batch 200/352 | loss 0.6781
epoch 2/20 | batch 300/352 | loss 0.6384
average loss of epoch 2/20 | : 0.6657441978935491
average validation loss of epoch 2/20 | : 0.6789862781763076
epoch 3/20 | batch 100/352 | loss 0.5219
epoch 3/20 | batch 200/352 | loss 0.6268
epoch 3/20 | batch 300/352 | loss 0.5010
average loss of epoch 3/20 | : 0.605044749988751
average validation loss of epoch 3/20 | : 0.5337179161608219
epoch 4/20 | batch 100/352 | loss 0.5181
epoch 4/20 | batch 200/352 | loss 0.4981
epoch 4/20 | batch 300/352 | loss 0.4691
average loss of epoch 4/20 | : 0.5583281002261422
average validation loss of epoch 4/20 | : 0.5169913224875927
epoch 5/20 | batch 100/352 | loss 0.4460
epoch 5/20 | batch 200/3

In [18]:
model.load_state_dict(torch.load("best_parameters.pth", map_location=device))
def predict_sentiment(text):
    
    with torch.no_grad():
        model.eval()

        text_tensor= torch.tensor(text_pipeline(text, vocab), dtype=torch.int64).unsqueeze(0).to(device)
        prediction= model(text_tensor)
        sentiment= 'positive' if prediction>0.5 else 'negative'

        return sentiment, prediction
    

reviews= [
    'The movie was really great',
    'I really loved the movie',
    'It was really bad',
    'it was okay at its best',
    'Its not bad, but I wouldnt watch it again',
    'I expected it to be terrible and somehow it wasnt',
    'The acting was great, the plot was painful.'
    
]

for review in reviews:
    sentiment,prediction= predict_sentiment(review)

    print(f"review: {review}")
    print(f"sentiment: {sentiment}")


review: The movie was really great
sentiment: positive
review: I really loved the movie
sentiment: positive
review: It was really bad
sentiment: negative
review: it was okay at its best
sentiment: positive
review: Its not bad, but I wouldnt watch it again
sentiment: negative
review: I expected it to be terrible and somehow it wasnt
sentiment: negative
review: The acting was great, the plot was painful.
sentiment: positive


In [19]:
total=0
correct=0
TrueP = 0
TrueN = 0
FalseP = 0
FalseN = 0

model.load_state_dict(torch.load("best_parameters.pth", map_location=device))

with torch.no_grad():
   model.eval()

   for label,text in test_loader:

      label= label.to(device).float()
      text=  text.to(device)

      output= model(text)

      probs= torch.sigmoid(output)
      preds= (probs>=0.5).float()

      total += label.size(0)
      correct += (preds.squeeze()==label).sum().item()

      TrueP += ((preds == 1) & (label == 1)).sum().item()
      TrueN += ((preds == 0) & (label == 0)).sum().item()
      FalseP += ((preds == 1) & (label == 0)).sum().item()
      FalseN += ((preds == 0) & (label == 1)).sum().item()

   conf_matrix = torch.tensor([[TrueP, FalseN],
                               [FalseP, TrueN]])
   accuracy= 100 * correct/total

print(accuracy)
print(conf_matrix)

86.32
tensor([[689436, 109604],
        [110540, 689460]])
